# Foco-LLM: 7B training-seed sensitivity (seed 43)

Post-hoc repeat of the fixed 320-step QLoRA protocol. Frozen arithmetic replication and fact-removal datasets remain unchanged. The repository is cloned into temporary storage so only experimental outputs are saved.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"This project requires Python 3.12 or 3.13; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
PINNED_COMMIT = "d403c12f6f25"
repository = Path("/kaggle/temp/foco-llm-seed43")
if not repository.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "feat/task-balanced-training",
            "--single-branch",
            "https://github.com/Mathwesm/foco-llm.git",
            str(repository),
        ],
        check=True,
    )
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=repository, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repository, check=True)
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)

In [ ]:
import hashlib
import json

required_sources = (
    "scripts/run_kaggle_selection.py",
    "scripts/run_blind_replication.py",
    "scripts/run_fact_ablation.py",
    "reports/2026-10-07/local-seed-replication-v2.json",
)
for relative_path in required_sources:
    if not (repository / relative_path).is_file():
        raise FileNotFoundError(f"Pinned repository is missing {relative_path}")
for dataset_name in ("blind-replication", "fact-ablation"):
    dataset_root = repository / "reports" / "2026-10-04" / dataset_name
    manifest = json.loads((dataset_root / "manifest.json").read_text(encoding="utf-8"))
    observed_sha256 = hashlib.sha256((dataset_root / "dataset.json").read_bytes()).hexdigest()
    if observed_sha256 != manifest["dataset_sha256"]:
        raise ValueError(f"Frozen dataset digest mismatch: {dataset_name}")
print("Pinned scripts and frozen datasets verified", flush=True)

In [ ]:
import json

TRAINING_SEED = 43
TRAINING_STEPS = 320
output = Path("/kaggle/working/foco-output-7b-seed43")
command = ["poetry", "run", "python", "scripts/run_kaggle_selection.py"]
subprocess.run(
    [*command, "prepare", "--model-size", "7b", "--output", str(output)], cwd=repository, check=True
)
training_dataset = output / "training-dataset.json"
reference_bundle = json.loads(
    (repository / "reports/2026-10-07/local-seed-replication-v2.json").read_text(encoding="utf-8")
)
expected_training_sha256 = reference_bundle["arms"][0]["training"]["manifest"]["dataset_sha256"]
if hashlib.sha256(training_dataset.read_bytes()).hexdigest() != expected_training_sha256:
    raise ValueError("Training dataset digest differs from the frozen protocol")
subprocess.run(
    [
        *command,
        "train",
        "--model-size",
        "7b",
        "--output",
        str(output),
        "--steps",
        str(TRAINING_STEPS),
        "--seed",
        str(TRAINING_SEED),
        "--checkpoint-every",
        "80",
    ],
    cwd=repository,
    check=True,
)
completed = sorted((output / "training").glob("*/summary.json"))
if len(completed) != 1:
    raise RuntimeError(f"Expected one completed seed-43 training run; found {len(completed)}")
training_root = completed[0].parent
training_manifest = json.loads((training_root / "manifest.json").read_text(encoding="utf-8"))
if (
    training_manifest["config"]["seed"] != TRAINING_SEED
    or training_manifest["config"]["steps"] != TRAINING_STEPS
):
    raise RuntimeError("Training configuration differs from frozen seed-43 protocol")
adapter = training_root / "step-0320"
if not (adapter / "complete.json").is_file():
    raise RuntimeError("Complete step-0320 adapter is missing")
print("Seed 43 training verified:", training_root, flush=True)

In [ ]:
def run_evaluation(script, dataset, target):
    """Evaluate one frozen dataset and return its saved summary path."""
    subprocess.run(
        [
            "poetry",
            "run",
            "python",
            f"scripts/{script}",
            "evaluate",
            "--dataset",
            str(repository / dataset),
            "--output",
            str(output / target),
            "--backend",
            "cloud",
            "--adapter",
            str(adapter),
            "--training-dataset",
            str(training_dataset),
        ],
        cwd=repository,
        check=True,
    )
    results = sorted((output / target).glob("*/summary.json"))
    if len(results) != 1:
        raise RuntimeError(f"Expected one complete {target} evaluation; found {len(results)}")
    return results[0]


replication_summary = run_evaluation(
    "run_blind_replication.py", "reports/2026-10-04/blind-replication/dataset.json", "replication"
)
ablation_summary = run_evaluation(
    "run_fact_ablation.py", "reports/2026-10-04/fact-ablation/dataset.json", "ablation"
)
summary = {
    "protocol": "post-hoc-7b-seed-sensitivity-v1",
    "training_seed": 43,
    "training_manifest": str(training_root / "manifest.json"),
    "replication": json.loads(replication_summary.read_text(encoding="utf-8")),
    "ablation": json.loads(ablation_summary.read_text(encoding="utf-8")),
}
(output / "comparison-summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
print("Seed 43 replication:", summary["replication"], flush=True)
print("Seed 43 ablation:", summary["ablation"], flush=True)